In [ ]:
# 📦 Step 1: Install required packages
# These packages include:
# - transformers: for using pre-trained language models
# - sentencepiece: required for some tokenizer backends
# - gradio: to build a simple web UI for user interaction
!pip install transformers sentencepiece gradio

In [ ]:
# 🧠 Step 2: Import libraries
from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio as gr
import torch

In [ ]:
# 🔍 Step 3: Load a public transformer model from Hugging Face
# Using google/flan-t5-large which is open-access and suitable for summarization/generation tasks
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForCausalLM.from_pretrained("google/flan-t5-large")

In [ ]:
# ✂️ Step 4a: Function to summarize medical text
def summarize_text(text):
    prompt = f"Summarize the following medical text:\n{text}\n\nSummary:"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=150)
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

In [ ]:
# 🧾 Step 4b: Function to generate a research article
def write_article(text):
    prompt = f"Based on the following information, write a research article:\n{text}\n\nArticle:"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=300)
    article = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return article

In [ ]:
# 🔒 Step 4c: Function to redact personally identifiable information (PHI)
def redact_phi(text):
    # Very basic redaction rule; in practice, use regex or trained NER models
    redacted_text = text.replace("Patient Name", "[REDACTED]")
    return redacted_text

In [ ]:
# 🔄 Step 5: Agent dispatching function based on user-selected task
def multi_agent_system(input_text, task):
    if task == "Summarization":
        return summarize_text(input_text)
    elif task == "Article Writing":
        return write_article(input_text)
    elif task == "PHI Redaction":
        return redact_phi(input_text)
    else:
        return "Invalid task selected." 

In [ ]:
# 🎛️ Step 6: Build the Gradio interface for interaction
with gr.Blocks() as demo:
    gr.Markdown("# 🧠 Medical Text Multi-Agent AI System")
    gr.Markdown("Paste medical notes, choose a task, and let the AI respond!")

    text_input = gr.Textbox(lines=10, label="Medical Text Input")
    task_selector = gr.Radio(["Summarization", "Article Writing", "PHI Redaction"], label="Select Task")
    output_box = gr.Textbox(label="AI Output")

    submit_btn = gr.Button("Run Agent")
    submit_btn.click(fn=multi_agent_system, inputs=[text_input, task_selector], outputs=output_box)

# 🚀 Step 7: Launch the UI
demo.launch()